In [2]:
import pandas as pd 
import pymysql 
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
raw_db = os.getenv("RAW_DB")
processed_db = os.getenv("PROCESSED_DB")

In [5]:
print(raw_db)
print(processed_db)

online_retail_raw
online_retail_processed


In [6]:
raw_conn = pymysql.connect(
    host=host,
    user=user,
    password=password,
    database=raw_db
)

print(" Raw Database Connected")

 Raw Database Connected


In [7]:
processed_conn = pymysql.connect(
    host=host,
    user=user,
    password=password,
    database=processed_db
)

print(" Processed Database Connected")

 Processed Database Connected


In [8]:
query = """ select * from etl_metadata order by id desc limit 1; """
metadata_df = pd.read_sql(query, processed_conn)
metadata_df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27072\385885294.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  metadata_df = pd.read_sql(query, processed_conn)


,id,last_processed_date,last_run_time,rows_processed,status,created_at
0,1,1900-01-01,2026-06-28 14:15:34,0,sucess,2026-06-28 14:15:34


In [9]:
last_processed_date = metadata_df.loc[0, "last_processed_date"]

print(last_processed_date)

1900-01-01 00:00:00


In [10]:
query = """ show columns from retail_raw; """
print(query)

 show columns from retail_raw; 


In [11]:
query =  f"""
SELECT *
FROM retail_raw
WHERE InvoiceDate > '{last_processed_date}';
"""

raw_df = pd.read_sql(query, raw_conn)
raw_df.head()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27072\2439830459.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  raw_df = pd.read_sql(query, raw_conn)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [12]:
print(raw_df.shape[0])
print(raw_df.head())

541909
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2  2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [13]:
raw_df.info()
raw_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [14]:
raw_df["InvoiceDate"] = pd.to_datetime(raw_df["InvoiceDate"])

In [15]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [16]:
raw_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [17]:
raw_df["Description"] = raw_df["Description"].fillna("Unknown Product")

In [18]:
raw_df["Description"].isnull().sum()

np.int64(0)

In [19]:
raw_df.duplicated().sum()

np.int64(5268)

In [20]:
raw_df = raw_df.drop_duplicates()

In [21]:
raw_df.shape[0]

536641

In [22]:
raw_df.duplicated().sum()

np.int64(0)

In [23]:
raw_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [24]:
(raw_df["UnitPrice"] <= 0).sum()

np.int64(2512)

In [25]:
raw_df[raw_df["UnitPrice"] <= 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,Unknown Product,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,Unknown Product,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,Unknown Product,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,Unknown Product,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,Unknown Product,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,Unknown Product,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,Unknown Product,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,Unknown Product,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,Unknown Product,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,Unknown Product,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [26]:
raw_df[raw_df["UnitPrice"] <= 0]["InvoiceNo"].str.startswith("C").value_counts()


InvoiceNo
False    2512
Name: count, dtype: int64

In [27]:
raw_df[raw_df["UnitPrice"] <= 0]["StockCode"].value_counts().head(10)

StockCode
23084    16
35965    13
20713    11
22501    10
22676     9
22084     9
21116     8
82583     8
22683     8
22627     8
Name: count, dtype: int64

In [28]:
raw_df.loc[
    (raw_df["StockCode"] == "23084") &
    (raw_df["UnitPrice"] == 0),
    ["InvoiceNo", "StockCode", "Description", "UnitPrice", "Quantity"]
]

,InvoiceNo,StockCode,Description,UnitPrice,Quantity
275365,560986,23084,Unknown Product,0.0,24
292271,562548,23084,Unknown Product,0.0,-84
301887,563359,23084,Unknown Product,0.0,20
314369,564633,23084,Unknown Product,0.0,400
314906,564667,23084,Unknown Product,0.0,-400
332824,566122,23084,temp adjustment,0.0,-484
336436,566327,23084,allocate stock for dotcom orders ta,0.0,4
337788,566476,23084,add stock to allocate online orders,0.0,2
337817,566478,23084,for online retail orders,0.0,1
340225,566615,23084,Unknown Product,0.0,344


In [29]:
raw_df = raw_df[raw_df["UnitPrice"] > 0]

In [30]:
print(raw_df.shape)

(raw_df["UnitPrice"] <= 0).sum()


(534129, 8)


np.int64(0)

In [31]:
(raw_df["Quantity"] == 0).sum()

np.int64(0)

In [37]:
raw_df = raw_df.where(pd.notnull(raw_df), None)

In [42]:
cursor = processed_conn.cursor()

cursor.execute("DROP TABLE IF EXISTS retail_clean")

processed_conn.commit()

print("Old table dropped successfully.")

Old table dropped successfully.


In [44]:
cursor.execute("SHOW CREATE TABLE retail_clean")

print(cursor.fetchone()[1])

CREATE TABLE `retail_clean` (
  `id` int NOT NULL AUTO_INCREMENT,
  `InvoiceNo` varchar(20) DEFAULT NULL,
  `StockCode` varchar(20) DEFAULT NULL,
  `Description` text,
  `Quantity` int DEFAULT NULL,
  `InvoiceDate` datetime DEFAULT NULL,
  `UnitPrice` decimal(10,2) DEFAULT NULL,
  `CustomerID` int DEFAULT NULL,
  `Country` varchar(50) DEFAULT NULL,
  `Revenue` decimal(12,2) DEFAULT NULL,
  PRIMARY KEY (`id`),
  KEY `idx_customer` (`CustomerID`),
  KEY `idx_date` (`InvoiceDate`),
  CONSTRAINT `retail_clean_chk_1` CHECK ((`Quantity` <> 0)),
  CONSTRAINT `retail_clean_chk_2` CHECK ((`UnitPrice` > 0))
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci


In [45]:
raw_df = raw_df.where(pd.notnull(raw_df), None)

In [46]:
raw_df["Revenue"] = raw_df["Quantity"] * raw_df["UnitPrice"]

In [48]:
data = list(
    raw_df[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country",
            "Revenue"
        ]
    ].itertuples(index=False, name=None)
)

In [69]:
raw_df = raw_df[raw_df["UnitPrice"] >= 0.01]

In [70]:
cursor.execute("DROP TABLE IF EXISTS retail_clean")
processed_conn.commit()

In [71]:
create_table_query = """
CREATE TABLE retail_clean (
    id INT AUTO_INCREMENT PRIMARY KEY,

    InvoiceNo VARCHAR(20),
    StockCode VARCHAR(20),
    Description TEXT,

    Quantity INT,
    InvoiceDate DATETIME,

    UnitPrice DECIMAL(10,3),

    CustomerID INT NULL,

    Country VARCHAR(50),

    Revenue DECIMAL(12,3),

    INDEX idx_customer (CustomerID),
    INDEX idx_date (InvoiceDate),

    CHECK (Quantity <> 0),
    CHECK (UnitPrice > 0)
)
"""

cursor.execute(create_table_query)
processed_conn.commit()

print("retail_clean recreated successfully.")

retail_clean recreated successfully.


In [72]:
raw_df["Revenue"] = raw_df["Quantity"] * raw_df["UnitPrice"]

In [73]:
import math

data = []

for row in raw_df[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
        "Revenue"
    ]
].itertuples(index=False, name=None):

    cleaned_row = tuple(
        None if (isinstance(value, float) and math.isnan(value)) else value
        for value in row
    )

    data.append(cleaned_row)

print(f"Total Rows Ready : {len(data)}")

Total Rows Ready : 534125


In [74]:
insert_query = """
INSERT INTO retail_clean
(
InvoiceNo,
StockCode,
Description,
Quantity,
InvoiceDate,
UnitPrice,
CustomerID,
Country,
Revenue
)

VALUES
(%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

In [76]:
batch_size = 1000

for i in range(0, len(data), batch_size):

    batch = data[i:i+batch_size]

    cursor.executemany(insert_query, batch)

    processed_conn.commit()

    if (i + batch_size) % 50000 == 0 or (i + batch_size) >= len(data):
        print(f"Inserted {min(i + batch_size, len(data))} rows")

Inserted 50000 rows
Inserted 100000 rows
Inserted 150000 rows
Inserted 200000 rows
Inserted 250000 rows
Inserted 300000 rows
Inserted 350000 rows
Inserted 400000 rows
Inserted 450000 rows
Inserted 500000 rows
Inserted 534125 rows


In [77]:
verify_query = """
SELECT COUNT(*) AS Total_Rows
FROM retail_clean;
"""

pd.read_sql(verify_query, processed_conn)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_27072\2083194753.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(verify_query, processed_conn)


,Total_Rows
0,534125
